In [1]:
"""First things first, we gotta design the Puzzle game state stuff
what do we need to do...
    -we need to be able to store states like 0,1,2,3,4,5,6,7,8
    which mean 0 1 2
               3 4 5
               6 7 8 
    -then we gotta be able to peform options L,R,U,D
    - these correspond to swaps with zero and some integer
    - imagine as list 0 - 8
     - imagine at INDEX 0, 3, 6 7, 8, 5, 2, 1, have less than 4 moves
     in fact only moves for positions are:
     0 :  Right, Down
     1: : Right, Left, Down
     2:   Left, Down
     3: Right, Down, UP
     4: ALL Possible
     5: UP, Down , Left
     6: UP, Right
     7:  Left, right, UP
     8:  Left, UP
     
         REMEMBER, ONLY 0 object can move"""
import time
import heapq

class EightBoard:
    
    def __init__(self, state: list,actions: tuple = None, instruct_list = None,depth = None, total_fn = 0, h = 0):
        self.state = state
        self.actions = actions
        self.instruct_list = instruct_list
        self.depth = depth
        self.h = h
        if self.instruct_list is None:
            self.instruct_list = []
        if self.actions is None: 
            self.Update_Actions()
        if self.depth is None:
            self.depth = 0
        self.total_fn = total_fn
        self.total_fn = self.Manhattan_plus_depth()

    def Copy(self):
        return EightBoard(self.state.copy(),self.actions,self.instruct_list.copy(),self.depth,self.total_fn,self.h)
    def Update_Actions(self):
        if 0 == self.state[0]:
            move_tup = ("Right","Down")
            self.actions = move_tup
        elif 0 == self.state[1]:
            move_tup = ("Right","Left","Down")
            self.actions = move_tup
        elif 0 == self.state[2]:
            move_tup = ("Left","Down")
            self.actions = move_tup
        elif 0 == self.state[3]:
            move_tup = ("Right","Up","Down")
            self.actions = move_tup
        elif 0 == self.state[4]:
            move_tup = ("Right","Left","Down","Up")
            self.actions = move_tup
        elif 0 == self.state[5]:
            move_tup = ("Up","Left","Down")
            self.actions = move_tup
        elif 0 == self.state[6]:
            move_tup = ("Right","Up")
            self.actions = move_tup
        elif 0 == self.state[7]:
            move_tup = ("Right","Left","Up")
            self.actions = move_tup
        elif 0 == self.state[8]:
            move_tup = ("Up","Left")
            self.actions = move_tup
    def Swap(self,x,y)->None:
        # Check if indices are within valid range
        if not (0 <= x < 9 and 0 <= y < 9):
            raise IndexError(f"Swap indices {x}, {y} out of range (0-8)")
        temp = self.state[x]
        self.state[x] = self.state[y]
        self.state[y] = temp
    def FindZero(self)->int:
        for i in range(len(self.state)):
            if self.state[i]==0:
                return i
    def Key(self):
        return tuple(self.state)
        
    def PossibleMove(self,move):
        zero_index = self.FindZero()
        new_board = self.Copy()
        if move not in self.actions:
            raise ValueError(f"Invalid move {move} for board {self.state}")
        elif move == "Up":
            new_board.Swap(zero_index,zero_index-3)
            
            
        elif move == "Down":
            new_board.Swap(zero_index,zero_index+3)   
            
            
        elif move =="Right":
            new_board.Swap(zero_index,zero_index +1)   
            
            
        elif move =="Left":
            new_board.Swap(zero_index, zero_index-1) 

        new_board.instruct_list.append(move)
        new_board.depth = self.depth +1
        new_board.Update_Actions()
        new_board.total_fn = new_board.Manhattan_plus_depth()
        return new_board
    def NextState(self,move):
        zero_index = self.FindZero()
        state_copy = self.state.copy()
        imagined_state = []
        if move not in self.actions:
            raise ValueError(f"Invalid move {move} for board {self.state}")
        elif move == "Up":
            self.SwapList(state_copy,zero_index,zero_index-3)
            
        elif move == "Down":
            self.SwapList(state_copy,zero_index,zero_index +3)
             
        elif move =="Right":  
            
            self.SwapList(state_copy,zero_index,zero_index+1)
        elif move =="Left":
            
            self.SwapList(state_copy,zero_index,zero_index -1)
        imagined_state = state_copy

        return tuple(imagined_state)
    def SwapList(self,some_list,x,y)->None:
        # Check if in grid
        if not (0 <= x < 9 and 0 <= y < 9):
            raise IndexError(f"Swap indices {x}, {y} out of range (0-8)")
        temp = some_list[x]
        some_list[x] = some_list[y]
        some_list[y] = temp

    def DisplayBoard(self):
        print(f'{self.state[0]} {self.state[1]} {self.state[2]}')
        print(f'{self.state[3]} {self.state[4]} {self.state[5]}')
        print(f'{self.state[6]} {self.state[7]} {self.state[8]}')
    def DisplayBoard2(self,some_list):
        print(f'{some_list[0]} {some_list[1]} {some_list[2]}')
        print(f'{some_list[3]} {some_list[4]} {some_list[5]}')
        print(f'{some_list[6]} {some_list[7]} {some_list[8]}')

    def PrintInstruct(self):
        print(self.instruct_list[::-1])
    def Perform_Instruction_List(self):
        state_copy = self.Copy()
        solution_list = state_copy.instruct_list
        some_node = None
        node_list = []
        for i in solution_list[::-1]:
            if i == "Up":
                state_copy = state_copy.PossibleMove("Down")
    
            elif i == "Down":
                state_copy = state_copy.PossibleMove("Up")
            elif i == "Right":
                state_copy = state_copy.PossibleMove("Left")
            elif i == "Left":
                state_copy = state_copy.PossibleMove("Right")
            node_list.append(state_copy)
        for i in (node_list[::-1]):
            i.DisplayBoard()
            print("to")
        self.DisplayBoard()
    def MisplacedTiles(self):
        goal = [0,1,2,3,4,5,6,7,8]
        
        count = 0
        for i in range(len(goal)):
            if self.state[i] != goal[i]:
                count+=1
        return count
    def GridLoc(self,some_state):
        
        return {
            some_state[0]: (0,0),
            some_state[1]: (0,1),
            some_state[2]: (0,2),
            some_state[3]: (1,0),
            some_state[4]: (1,1),
            some_state[5]:(1,2),
            some_state[6]:(2,0),
            some_state[7]:(2,1),
            some_state[8]:(2,2)
        }
    def Manhattan_Distance(self):
        man_dict = dict()
        final_sum = 0
        state_pos = self.GridLoc(self.state)
        goal_pos = self.GridLoc([0,1,2,3,4,5,6,7,8])
        
        for key in goal_pos:
            if key != 0:
                man_dict[key] = abs(goal_pos[key][0] - state_pos[key][0]) + abs(goal_pos[key][1] - state_pos[key][1])
        for key in man_dict:
            final_sum += man_dict[key]
        return final_sum
    def Manhattan_plus_depth(self):
        if self.h == 0:
            if self.depth == 0:
                return self.MisplacedTiles()
            else:
                return self.MisplacedTiles() +self.depth
        else:
            if self.depth ==0:
                return self.Manhattan_Distance()
            
            else:
                return self.Manhattan_Distance() + self.depth
    

    

        
        
                                                                          

                                                            

        
        
                
            
                
        
        
                
                
        
    
    

    
            
    

In [2]:
#implementation of function "A Star Manhattan Distance"
"""A-Star Implementation"""
import heapq

def A_Star_Search_1(genNode):
    start = time.time()
    cueoo = [] # create queue
    somedict = dict() # create dictionary
    steps = 0
    counter = 0
    goal = (0,1,2,3,4,5,6,7,8)
    genNode.h =1
    
    
    heapq.heappush(cueoo, (genNode.total_fn,counter,genNode)) # append node to priority queue

    first_in = 0,0,EightBoard([1,2,4,5,6,3,7,8,0]) # just to create board_object to be reassigned later

    while cueoo:
        counter+=1 # this increments to help stop tie breaks
        first_in = heapq.heappop(cueoo) # pop from priority queue


    
        key = first_in[2].Key()
        if key in somedict and somedict[key] <= first_in[2].depth: # if dict depth is greater than current path depth
            continue

       

            
        
        if key == goal:
            end = time.time()
            time_taken = end - start
            steps = first_in[2].depth
            return [first_in[2],time_taken,steps]

        if key not in somedict:
            somedict[key] = first_in[2].depth # here we need to add to dictioanry, using some key name
            for i in first_in[2].actions: # here we loop through actions list
                child_state = first_in[2].NextState(i) 
                if child_state in somedict and somedict[child_state] <= first_in[2].depth +1: # handling revisiting but with different depths
                    continue
                else:
                    child_node = first_in[2].PossibleMove(i)
                    
                    heapq.heappush(cueoo,(child_node.total_fn,counter,child_node)) # append to priority queue
                    counter+=1
        

        else:
            
            for i in first_in[2].actions: # here we loop through actions list
                child_state = first_in[2].NextState(i) 
                if child_state in somedict and somedict[child_state]<= first_in[2].depth +1: # handling revisiting but with different depths
                    continue
                else:
                    child_node = first_in[2].PossibleMove(i)
                    
                    heapq.heappush(cueoo,(child_node.total_fn,counter,child_node)) # append to priority queue
                    counter+=1
        somedict[key] = first_in[2].depth
    
           
            
            
            
        
            
            
    end = time.time()
    time_taken = end - start
    steps = first_in[2].depth

    return [first_in[2],time_taken,steps]

fin = open('Input8PuzzleCases-1.txt')
def CreateInputList(file_object):   
    input_list = []
    for i in file_object:

        # Split into a list of strings
        i = i.strip()
        string_list = i.split(",")
        for j in range(len(string_list)):
            string_list[j] = int(string_list[j])
        input_list.append(string_list)
    return input_list
puzzinputs = CreateInputList(fin)

def AverageTime_and_S_Taken(func,inputs,n):
    sum1= 0
    sum2 = 0
    count = 0
    for i in inputs:
        if count <n:
            some_node = EightBoard(i)
            current_sol = func(some_node)
            sum1 +=current_sol[1]
            sum2+= current_sol[2]
            count+=1
            print(count)
        else:
            break
    average1 = sum1/n
    average2 = sum2/n
    return [average1,average2]




A_Star1_stats = AverageTime_and_S_Taken(A_Star_Search_1,puzzinputs,100)
# this function will take aobut 6 and 1/2 minutes to run(for 100 iterations)
#average is about 4 seconds for solution

print(A_Star1_stats)



1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
[0.15286150455474853, 22.33]


In [3]:
#implementation of function "A Star Misplaced"
"""A-Star Implementation"""
import heapq

def A_Star_Search_0(genNode):
    start = time.time()
    cueoo = [] # create queue
    somedict = dict() # create dictionary
    steps = 0
    counter = 0
    goal = (0,1,2,3,4,5,6,7,8)
    
    
    
    heapq.heappush(cueoo, (genNode.total_fn,counter,genNode)) # append node to priority queue

    first_in = 0,0,EightBoard([1,2,4,5,6,3,7,8,0]) # just to create board_object to be reassigned later

    while cueoo:
        counter+=1 # this increments to help stop tie breaks
        first_in = heapq.heappop(cueoo) # pop from priority queue


    
        key = first_in[2].Key()
        if key in somedict and somedict[key] <= first_in[2].depth: # if dict depth is greater than current path depth
            continue

       

            
        
        if key == goal:
            end = time.time()
            time_taken = end - start
            steps = first_in[2].depth
            return [first_in[2],time_taken,steps]

        if key not in somedict:
            somedict[key] = first_in[2].depth # here we need to add to dictioanry, using some key name
            for i in first_in[2].actions: # here we loop through actions list
                child_state = first_in[2].NextState(i) 
                if child_state in somedict and somedict[child_state] <= first_in[2].depth +1: # handling revisiting but with different depths
                    continue
                else:
                    child_node = first_in[2].PossibleMove(i)
                    
                    heapq.heappush(cueoo,(child_node.total_fn,counter,child_node)) # append to priority queue
                    counter+=1
        

        else:
            
            for i in first_in[2].actions: # here we loop through actions list
                child_state = first_in[2].NextState(i) 
                if child_state in somedict and somedict[child_state]<= first_in[2].depth +1: # handling revisiting but with different depths
                    continue
                else:
                    child_node = first_in[2].PossibleMove(i)
                    
                    heapq.heappush(cueoo,(child_node.total_fn,counter,child_node)) # append to priority queue
                    counter+=1
        somedict[key] = first_in[2].depth
    
           
            
            
            
        
            
            
    end = time.time()
    time_taken = end - start
    steps = first_in[2].depth

    return [first_in[2],time_taken,steps]

fin = open('Input8PuzzleCases-1.txt')
def CreateInputList(file_object):   
    input_list = []
    for i in file_object:

        # Split into a list of strings
        i = i.strip()
        string_list = i.split(",")
        for j in range(len(string_list)):
            string_list[j] = int(string_list[j])
        input_list.append(string_list)
    return input_list
puzzinputs = CreateInputList(fin)

def AverageTime_and_S_Taken(func,inputs,n):
    sum1= 0
    sum2 = 0
    count = 0
    for i in inputs:
        if count <n:
            some_node = EightBoard(i)
            current_sol = func(some_node)
            sum1 +=current_sol[1]
            sum2+= current_sol[2]
            count+=1
            print(count)
        else:
            break
    average1 = sum1/n
    average2 = sum2/n
    return [average1,average2]




A_Star0_stats= AverageTime_and_S_Taken(A_Star_Search_0,puzzinputs,100)
# this function will take aobut 6 and 1/2 minutes to run(for 100 iterations)
#average is about 4 seconds for solution

print(A_Star0_stats)



1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
[0.8365446925163269, 22.33]


In [4]:
#implementation of function "print_result(result)"
def print_result():
    some_node= EightBoard([8, 7, 5, 4, 1, 2, 3, 0, 6])
    
    solution = A_Star_Search_1(some_node)[0]
    
    solution.Perform_Instruction_List()
    print("        Average Steps | Average Time")
    print(f'A*0:   {A_Star0_stats[1]}            |   {A_Star0_stats[0]}')
    print(f'A*1:   {A_Star1_stats[1]}            |   {A_Star1_stats[0]}')
    
print_result()

8 7 5
4 1 2
3 0 6
to
8 7 5
4 0 2
3 1 6
to
8 7 5
4 2 0
3 1 6
to
8 7 0
4 2 5
3 1 6
to
8 0 7
4 2 5
3 1 6
to
8 2 7
4 0 5
3 1 6
to
8 2 7
4 1 5
3 0 6
to
8 2 7
4 1 5
3 6 0
to
8 2 7
4 1 0
3 6 5
to
8 2 0
4 1 7
3 6 5
to
8 0 2
4 1 7
3 6 5
to
8 1 2
4 0 7
3 6 5
to
8 1 2
0 4 7
3 6 5
to
0 1 2
8 4 7
3 6 5
to
1 0 2
8 4 7
3 6 5
to
1 4 2
8 0 7
3 6 5
to
1 4 2
0 8 7
3 6 5
to
1 4 2
3 8 7
0 6 5
to
1 4 2
3 8 7
6 0 5
to
1 4 2
3 0 7
6 8 5
to
1 4 2
3 7 0
6 8 5
to
1 4 2
3 7 5
6 8 0
to
1 4 2
3 7 5
6 0 8
to
1 4 2
3 0 5
6 7 8
to
1 0 2
3 4 5
6 7 8
to
0 1 2
3 4 5
6 7 8
        Average Steps | Average Time
A*0:   22.33            |   0.8365446925163269
A*1:   22.33            |   0.15286150455474853
